# Evidence-driven agent teams

**Draft Cookbook contribution · Sophia Cole / ColeWorld · MIT licensed; see LICENSE.**

Build an application-owned scheduler that creates a targeted specialist only when evidence conflicts, stops unproductive work, and preserves dissent instead of voting it away.

This notebook uses fictional release records and a deterministic offline backend. It makes **no API calls** and requires no credentials. It is a draft contribution that has not been accepted or endorsed by OpenAI.

## 1. Why branch on evidence?

A fixed team can query every source for every question. That is easy to reason about, but it does not distinguish agreement, a targeted conflict, and a conflict that remains genuinely unresolved.

Here the application starts two evidence readers, groups records by `(claim, scope)`, then creates a specialist task only if the admitted records disagree. A configured routing table says where to look; the observed evidence determines whether a task is instantiated, which conflict it addresses, and when branching stops.

The [parallel agents Cookbook example](https://developers.openai.com/cookbook/examples/agents_sdk/parallel_agents) demonstrates independent parallel analyses and synthesis. This recipe instead studies **adaptive scheduling and evidence retention**, not parallel execution. Every call here is sequential. A branch is a specialist assignment, not an OS process or an API-hosted agent team.

### Setup

Use Python 3.10+ and keep this notebook beside `evidence_teams.py`. The walkthrough and unit tests use only the standard library. `requirements.txt` supplies `nbformat` for the separate validation script.

No SDK, API key, install command, or network-enabled client is executed by these cells.

In [ ]:
from dataclasses import asdict
from types import SimpleNamespace
from unittest.mock import Mock
import json

from evidence_teams import (
    Finding, Limits, Probe, ResponsesSpecialist, Scheduler,
    fixture_specialist, fixtures,
)

cases = fixtures()
topic = ("active_schema", "fictional-service/release-42")
print("Offline fixtures:", ", ".join(cases))

## 2. Separate evidence from decisions

Each immutable record has an ID, source identity, normalized claim, scope, value, and excerpt. A specialist returns **catalog IDs**, not arbitrary new facts. The scheduler validates the entire response against the task's source and scope before admitting any records.

Deduplication uses `(source, claim, scope, value)`: a reworded copy from one source is not independent evidence. The admitted ledger is append-only. An outcome can change, but older dissent does not disappear.

For arbitrary documents, trusted source retrieval and claim normalization are additional work. These fixtures begin with normalized records so we can isolate the scheduling policy.

In [ ]:
conflict_case = cases["conflict"]
for record in conflict_case.catalog[:2]:
    print(json.dumps(asdict(record), sort_keys=True))
assert conflict_case.catalog[0].scope == conflict_case.catalog[1].scope
assert conflict_case.catalog[0].value != conflict_case.catalog[1].value

### An explicit policy, not a majority vote

- `supported`: every admitted record agrees. This is not a truth or coverage guarantee.
- `resolved_by_policy`: the evidence disagrees, but configured authoritative sources have one value. Dissent is retained.
- `unresolved`: there is no unique policy-preferred value. More votes do not settle it.
- `unknown`: no valid evidence has been admitted.

In the fictional release example, application policy treats `signed_manifest` as authoritative. That name is a fixture label, not proof of a verified signature. Real source authority must be independently verified and scoped by application code, never asserted by a model.

## 3. Agreement: do not create a specialist

Both initial records say `v3`. No contradiction is observed, so the extra probes are not instantiated. An unqueried source could still disagree; this stopping rule deliberately trades source coverage for less work, and must be evaluated for the real domain.

In [ ]:
agreement = cases["agreement"].run()
print(asdict(agreement.decisions[topic]))
print("calls:", agreement.calls, "created branches:", agreement.created_branches)
assert agreement.calls == 2
assert agreement.created_branches == 0
assert agreement.decisions[topic].status == "supported"

## 4. Conflict: create a focused task, retain dissent

`release_note` says `v3`, but `ops_note` says `v2` for the same release. The scheduler creates a `manifest_reconciler` assignment with the exact claim, scope, source target, and IDs of the conflicting records.

The manifest record supplies `v3`. The application policy resolves this fixture, so the archive task is never created. The `v2` record remains in the ledger and in `dissent_ids`.

In [ ]:
conflict = cases["conflict"].run()
for branch in conflict.branches:
    print(branch.task.role, branch.status, branch.task.context_ids, branch.accepted_ids)
print("Decision:", asdict(conflict.decisions[topic]))
assert (conflict.calls, conflict.created_branches) == (3, 1)
assert conflict.branches[-1].task.context_ids == ("c1", "c2")
assert conflict.decisions[topic].dissent_ids == ("c2",)
assert "c2" in {record.id for record in conflict.evidence}
assert all(branch.task.role != "archive_checker" for branch in conflict.branches)

## 5. Unresolved: do not manufacture consensus

The third fixture contains two conflicting records from the policy-preferred source. The archive adds another `v3` observation, but that does not remove the contradiction. Once the permitted probes are exhausted, the correct result under this policy is an explicit unresolved outcome.

In [ ]:
unresolved = cases["unresolved"].run()
print("Decision:", asdict(unresolved.decisions[topic]))
print("Stopped because:", unresolved.termination, "review needed:", unresolved.needs_review)
assert unresolved.decisions[topic].value is None
assert unresolved.decisions[topic].status == "unresolved"
assert {"u3", "u4"} <= set(unresolved.decisions[topic].dissent_ids)
assert unresolved.termination == "exhausted"
assert unresolved.needs_review

## 6. Compare with a predeclared fixed-fanout control

The fixed control runs both readers and both configured probes without waiting for a conflict. Both modes use the same catalog, backend, decision rule, and total-call budget. Both execute sequentially; this is not a concurrency benchmark.

The only measured quantity below is logical specialist-call count on three hand-built fixtures. No live model, token usage, dollar cost, answer-quality improvement, or latency advantage is measured. The control can inspect more sources and may find contradictions the adaptive path misses.

In [ ]:
comparison = []
for name, case in cases.items():
    adaptive = case.run()
    fixed = case.run(adaptive=False)
    row = {
        "fixture": name,
        "adaptive_calls": adaptive.calls,
        "fixed_calls": fixed.calls,
        "adaptive_status": adaptive.decisions[topic].status,
        "fixed_status": fixed.decisions[topic].status,
    }
    comparison.append(row)
    print(json.dumps(row))
assert [row["adaptive_calls"] for row in comparison] == [2, 3, 4]
assert [row["fixed_calls"] for row in comparison] == [4, 4, 4]

## 7. Bound work without hiding uncertainty

`max_calls` includes initial reads and failures. `max_branches` bounds newly created adaptive assignments; `max_depth` bounds consecutive probes for one topic. Fixed mode uses the call cap, not the adaptive branch/depth caps.

A budget stop is not a resolution. A failed or empty initial reader also sets `needs_review`, even if the other reader supports a value. `termination="no_open_conflicts"` only describes the scheduler's stop condition, not research completeness. The backend must enforce its own timeout because a synchronous Python callable cannot be preempted here.

In [ ]:
limited = cases["unresolved"].run(limits=Limits(max_calls=8, max_branches=1, max_depth=3))
print("Stopped because:", limited.termination, "calls:", limited.calls)
assert limited.termination == "branch_limit"
assert limited.calls == 3
assert limited.decisions[topic].status == "unresolved"
assert limited.needs_review

def one_empty_reader(task, allowed):
    if task.sources == ("ops_note",):
        return Finding()
    return fixture_specialist(task, allowed)

partial = cases["agreement"].run(specialist=one_empty_reader)
assert partial.decisions[topic].status == "supported"
assert partial.needs_review
print("One empty reader:", partial.termination, "review needed:", partial.needs_review)

### Redundant and exhausted branches are terminal

A task's canonical key prevents the same assignment from running twice. A different specialist returning only previously admitted facts is `redundant`; returning no evidence is `exhausted`. Neither branch is automatically retried. Unresolved dissent remains visible, and another permitted source may still be investigated.

In [ ]:
case = cases["unresolved"]
probes = {topic: (
    Probe("repeat_reader", ("release_note",)),
    Probe("repeat_reader", ("release_note",)),  # identical assignment: skipped
    Probe("empty_archive", ("missing_source",)),
)}
terminal = Scheduler(case.catalog, probes, case.authoritative_sources).run(case.seeds, fixture_specialist)
print([branch.status for branch in terminal.branches])
assert [branch.status for branch in terminal.branches[2:]] == ["redundant", "duplicate_task", "exhausted"]
assert terminal.calls == 4
assert terminal.termination == "exhausted"
assert terminal.decisions[topic].status == "unresolved"

## 8. Optional Responses transport, tested with a fake client

`ResponsesSpecialist` uses the documented [Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs) JSON-schema request shape. It accepts an injected client and explicit model name, and rejects malformed, incomplete, refused, or out-of-scope results. No tools are supplied.

The next cell uses a local mock. It does not import the OpenAI SDK, read credentials, or contact a service. Passing a real client would incur API usage and transmit records, so a live experiment is a separate authorized step. See the README for timeout, retry, permissions, and usage-budget requirements.

In [ ]:
fake_response = SimpleNamespace(
    status="completed",
    output=[],
    output_text=json.dumps({"evidence_ids": ["a1"], "note": "Offline fake response"}),
)
fake_client = SimpleNamespace(responses=SimpleNamespace(create=Mock(return_value=fake_response)))
backend = ResponsesSpecialist(fake_client, model="offline-fixture-model")
finding = backend(cases["agreement"].seeds[0], (cases["agreement"].catalog[0],))
assert finding.evidence_ids == ("a1",)
request = fake_client.responses.create.call_args.kwargs
assert request["text"]["format"]["strict"] is True
assert request["store"] is False
assert "tools" not in request
print("Fake-client contract passed; no API call made.")

## 9. What this does and does not establish

The unit suite covers task and evidence deduplication, branch/call/depth budgets, failed specialists, atomic citation validation, empty readers, unresolved contradictions, and the fake Responses boundary. Run it with `python -m unittest discover -s tests -v`.

The local validation script checks the notebook schema, compiles its code, and runs cells sequentially in one Python namespace. This is **not a full Jupyter-kernel or notebook-rendering test**. Live SDK/service compatibility and model behavior are untested.

Important limitations:

- A model can omit a conflicting allowed record. Citation allowlisting and Structured Outputs prevent neither incomplete selection nor misleading interpretation.
- Agreement among admitted records does not guarantee correctness or completeness.
- The fictional source-precedence rule needs a domain-specific justification before real use.
- The scheduler only explores configured probes and routes observed contradictions, not all information gaps.
- This small synthetic comparison establishes control flow, not empirical superiority.

Before a live evaluation, add held-out conflicting and misleading sources, independent answer labels, source-coverage and omitted-dissent metrics, actual token/cost/latency measurements, and human review for unresolved outcomes. Keep retrieval permissions and source verification outside model-generated instructions.